In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from mapie.subsample import BlockBootstrap
import matplotlib.pyplot as plt

from var import DATA_OUT, START_DATE
from scintill_ai.conformal import enbpi_ts_regressor_predict, aci_ts_regressor_predict

## Data

In [ ]:
df = pd.read_pickle(Path(DATA_OUT, 'df.pickle'))

df['s4_mean_lag10'] = df['s4_mean'].shift(10)
df['n_sat_lag10'] = df['n_sat'].shift(10)

In [ ]:
X_cols = [
    'h_tmk',
    'f10.7_adj',
    'sza',
    's4_mean_lag10',
    'n_sat_lag10',
]

y_col = 's4_mean'

X_train, X_test = df.loc['2024-01-01':'2024-04-01', X_cols].copy(), df.loc['2024-04-02':'2024-04-06', X_cols].copy()
y_train, y_test = df.loc['2024-01-01':'2024-04-01', y_col].copy().fillna(0), df.loc['2024-04-02':'2024-04-06', y_col].copy().fillna(0)

## Random Forest regressor

In [ ]:
rf = RandomForestRegressor(
    max_depth=4, n_estimators=20, random_state=42, # OPTIMISE!
)

## Conformal Prediction

In [ ]:
cv = BlockBootstrap(
    n_resamplings=10, n_blocks=10, overlapping=False, random_state=42,
)

In [ ]:
# ALPHAS = [1 - .80, 1 - .90, 1 - .95]
# GAP = 30

### EnbPI (*without* update of residuals)

In [ ]:
enbpi_res = enbpi_ts_regressor_predict(
    model=rf, cv=cv, train_data=(X_train, y_train), test_data=(X_test, y_test)
)

### ACI (*without* update of residuals)

In [ ]:
aci_res = aci_ts_regressor_predict(
    model=rf,
    cv=cv,
    # alpha_list=ALPHAS,
    # gap=GAP,
    gamma=0.05,
    train_data=(X_train, y_train),
    test_data=(X_test, y_test),
)

### EnbPI (*with* update of residuals)

### ACI (*with* update of residuals)

## Plot

In [ ]:
plot_dict = aci_res

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, label="Actual (test)", c="C1")
ax.plot(
    y_test.index,
    plot_dict[0]["y_pred"],
    lw=1,
    c="C2",
    label="Forecast"
)

for result_ in plot_dict:
    y_pis = result_["y_pis"]
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.2,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['mean_width']:.2f} – CWC {result_['cwc']:.2f})",
    )

ax.set_title('EnbPI, with partial_fit', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.legend(prop={'size': 10}, loc='upper right', frameon=False)
ax.set_xlim(y_test.index[0], y_test.index[-1])
ax.set_ylim(0, 0.7)

plt.show()